In [1]:
# Install necessary libraries
!pip install pdfplumber spacy

# Download the English language model for Spacy
!python -m spacy download en_core_web_sm

print("Tools installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 76.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 81.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Tools installed successfully!


In [1]:
import pdfplumber
import spacy

# Load the AI model
nlp = spacy.load("en_core_web_sm")

# Define the keywords that suggest risk
RISK_KEYWORDS = [
    "penalty", "termination", "liable", "breach", "indemnify",
    "confidential", "non-compete", "waiver", "fine", "sued"
]

def analyze_contract(pdf_path):
    full_text = ""

    # 1. Extract text from PDF
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            full_text += page.extract_text() + "\n"

    # 2. Process text with AI
    doc = nlp(full_text)

    print("----- ANALYSIS REPORT -----\n")

    # 3. Identify Risky Sentences
    print("⚠️ POTENTIAL RISKS FOUND:\n")
    found_risks = False
    for sent in doc.sents: # Split into sentences
        sentence_text = sent.text.lower()
        for keyword in RISK_KEYWORDS:
            if keyword in sentence_text:
                print(f"[RISK: {keyword.upper()}] -> {sent.text.strip()}\n")
                found_risks = True

    if not found_risks:
        print("No major risk keywords found. Contract seems safe.")

    # 4. Extract Key Entities (Dates, Money, Orgs)
    print("\n----- KEY DETAILS -----")
    for ent in doc.ents:
        if ent.label_ in ["DATE", "MONEY", "ORG", "LAW"]:
            print(f"{ent.label_}: {ent.text}")

    return full_text

In [2]:
text = analyze_contract("rental1.pdf")

----- ANALYSIS REPORT -----

⚠️ POTENTIAL RISKS FOUND:

[RISK: TERMINATION] -> The said amount of the Security deposit shall be refunded by the Owner to
the Tenant at the time of handing over possession of the demised premises by the Tenant upon
expiry or sooner termination of this Rent after adjusting any dues (if any) or cost towards
damages caused by the negligence of the Tenant or the person he is responsible for, normal wear
& tear and damages due to act of god exempted.

[RISK: TERMINATION] -> In case the Owner fails to refund the security
deposit to the Tenant on early termination or expiry of the Rent agreement, the Tenant is entitled
to hold possession of the Rented premises, without payment of rent and/or any other charges
whatsoever, till such time the Owner refunds the security deposit to the Tenant.

[RISK: TERMINATION] -> On termination of the tenancy or earlier, the Tenant will be entitled to
remove such equipment and restore the changes made, if any, to the original sta

In [5]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 91.3 MB/s eta 0:00:00


In [11]:
!pip install streamlit pdfplumber spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 68.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [1]:
%%writefile app.py
import streamlit as st
import pdfplumber
import spacy

# Load the AI model (we installed this in Step 2)
nlp = spacy.load("en_core_web_sm")

# Risk Keywords
RISK_KEYWORDS = [
    "penalty", "termination", "liable", "breach", "indemnify",
    "confidential", "non-compete", "waiver", "fine", "sued"
]

# App Configuration
st.set_page_config(page_title="Legal Risk Scanner", layout="wide")

# --- SIDEBAR ---
st.sidebar.header("About")
st.sidebar.info("This app scans PDF contracts for risky keywords using NLP.")

# --- MAIN PAGE ---
st.title("📜 The Fine Print Guardian")
st.write("Upload a contract PDF to analyze potential risks.")

# FILE UPLOADER
uploaded_file = st.file_uploader("Upload your Contract PDF here", type="pdf")

if uploaded_file is not None:
    st.success("File uploaded successfully!")

    with st.spinner('Reading and analyzing...'):
        # Extract text
        full_text = ""
        try:
            with pdfplumber.open(uploaded_file) as pdf:
                for page in pdf.pages:
                    text = page.extract_text()
                    if text:
                        full_text += text + "\n"

        except Exception as e:
            st.error(f"Error reading PDF: {e}")

        if full_text.strip() != "":
            # Process with AI
            doc = nlp(full_text)

            st.subheader("📄 Analysis Results")

            col1, col2 = st.columns(2)

            # RISK ANALYSIS
            with col1:
                st.markdown("### ⚠️ Risky Clauses Found")
                risk_found = False
                for sent in doc.sents:
                    sentence_text = sent.text.lower()
                    for keyword in RISK_KEYWORDS:
                        if keyword in sentence_text:
                            st.warning(f"**Keyword:** {keyword.upper()}")
                            st.text(sent.text)
                            risk_found = True
                            break # Avoid duplicate warnings for same sentence

                if not risk_found:
                    st.success("✅ No major risks detected!")

            # ENTITY EXTRACTION
            with col2:
                st.markdown("### 🗂️ Key Details")
                for ent in doc.ents:
                    if ent.label_ in ["DATE", "MONEY", "ORG"]:
                        st.write(f"**{ent.label_}:** {ent.text}")
        else:
            st.error("The PDF seems empty or is an image.")

Overwriting app.py


In [5]:
!pip install cloudflared

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 3.8 MB/s eta 0:00:00
  Created wheel for cloudflared: filename=cloudflared-1.0.0.2-py3-none-any.whl size=2983 sha256=cf61186066a04dff5b0e1ebd1c2767067473302b0e8775640e91b8c22b355ac6
  Stored in directory: /root/.cache/pip/wheels/5b/ec/09/c3bcd3470be046ec77a9c0cb9d8bb6ceed49c831460878ab0a
Successfully built cloudflared


In [7]:
# Download the cloudflared binary
!curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o cloudflared

# Give it permission to run
!chmod +x cloudflared

# Move it to the system folder so we can use it easily
!mv cloudflared /usr/local/bin/

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 37.4M  100 37.4M    0     0  46.9M      0 --:--:-- --:--:-- --:--:--  134M


In [ ]:
# Start Streamlit and create the tunnel
!streamlit run app.py & cloudflared tunnel --url http://localhost:8501

2026-02-28T06:03:14Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-02-28T06:03:14Z INF Requesting new quick Tunnel on trycloudflare.com...



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://34.11.43.218:8502

2026-02-28T06:03:18Z INF +--------------------------------------------------------------------------------------------+
2026-02-28T06:03:18Z INF |